# CrossEntropyLoss 手动实现与公式讲解

这个 notebook 用本项目 `train.py` 中的翻译训练场景讲解 `CrossEntropyLoss`。

在你的训练代码里，模型输出和标签的形状是：

- `logits.shape = [batch_size, tgt_len, tgt_vocab_size]`
- `tgt_output.shape = [batch_size, tgt_len]`

`CrossEntropyLoss` 需要：

- `input.shape = [N, num_classes]`
- `target.shape = [N]`

所以 `train.py` 中会把前两个维度展平：

```python
vocab_size = logits.size(-1)
loss = criterion(logits.reshape(-1, vocab_size), tgt_output.reshape(-1))
```

本 notebook 会解释为什么这么写，并用完全手动实现验证 PyTorch 的 `nn.CrossEntropyLoss`。

## 1. CrossEntropyLoss 到底吃什么

对一个分类样本，模型输出一组 logits：

$$z = [z_1, z_2, \dots, z_C]$$

其中 `C` 是类别数。翻译任务里，`C` 就是目标语言词表大小 `tgt_vocab_size`。

如果真实类别是 `y`，交叉熵损失是：

$$Loss(z, y) = -\log \frac{e^{z_y}}{\sum_{j=1}^{C} e^{z_j}}$$

也可以写成更稳定的形式：

$$Loss(z, y) = \log \sum_{j=1}^{C} e^{z_j} - z_y$$

这说明 PyTorch 的 `nn.CrossEntropyLoss` 内部等价于：

```text
LogSoftmax + NLLLoss
```

因此要注意两个关键点：

1. 输入应该是 raw logits，不要先手动 softmax。
2. target 应该是类别 id，比如 `3`，不是 one-hot 向量。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

print("torch version:", torch.__version__)

## 2. 单个 token 的交叉熵手算

先看最小情况：一个位置预测 4 个类别。

- `logits.shape = [4]`
- `target = 2`

如果 `target = 2`，说明正确答案是第 2 类。PyTorch 中类别下标从 0 开始，所以第 2 类是第三个位置。

In [ ]:
single_logits = torch.tensor([1.0, 2.0, 4.0, 0.5])
single_target = torch.tensor(2)

print("single_logits.shape:", single_logits.shape)
print("single_target:", single_target.item())

In [ ]:
# 直观但不够稳定的写法：先 softmax，再取正确类别概率，再取 -log。
probs = torch.softmax(single_logits, dim=-1)
loss_naive = -torch.log(probs[single_target])

print("probs:", probs)
print("correct class prob:", probs[single_target].item())
print("loss_naive:", loss_naive.item())

## 3. 数值稳定版本：logsumexp

实际实现通常不用 `softmax` 后再 `log`，而是用 `logsumexp`：

$$Loss(z, y) = \log \sum_{j=1}^{C} e^{z_j} - z_y$$

PyTorch 的 `torch.logsumexp` 会处理大 logits，避免 `exp(logit)` 溢出。

In [ ]:
loss_stable = torch.logsumexp(single_logits, dim=-1) - single_logits[single_target]

print("loss_stable:", loss_stable.item())
print("same as naive:", torch.allclose(loss_naive, loss_stable))

## 4. 和 PyTorch `nn.CrossEntropyLoss` 对齐

`CrossEntropyLoss` 的 input 需要至少是二维：

- input: `[N, C]`
- target: `[N]`

所以单个样本也要变成：

- `single_logits.unsqueeze(0).shape = [1, 4]`
- `single_target.unsqueeze(0).shape = [1]`

In [ ]:
criterion = nn.CrossEntropyLoss()
torch_loss = criterion(single_logits.unsqueeze(0), single_target.unsqueeze(0))

print("torch_loss:", torch_loss.item())
print("same as stable:", torch.allclose(torch_loss, loss_stable))

## 5. 完全手动实现 CrossEntropyLoss

下面实现一个手写版本，支持：

- `logits.shape = [N, C]`
- `target.shape = [N]`
- `ignore_index`
- `reduction = "none" / "sum" / "mean"`

核心公式仍然是：

$$Loss_i = \log \sum_{j=1}^{C} e^{z_{i,j}} - z_{i,y_i}$$

In [ ]:
def cross_entropy_manual(logits, target, ignore_index=None, reduction="mean"):
    """完全手动实现 CrossEntropyLoss，不调用 nn.CrossEntropyLoss。

    参数：
        logits: [N, C]，模型原始输出，不要先 softmax
        target: [N]，每个样本的正确类别 id
        ignore_index: 如果 target 等于这个值，则该位置不参与 loss
        reduction: "none" / "sum" / "mean"

    返回：
        根据 reduction 返回逐位置 loss、总和或平均值。
    """
    if logits.dim() != 2:
        raise ValueError(f"logits must be [N, C], got {tuple(logits.shape)}")
    if target.dim() != 1:
        raise ValueError(f"target must be [N], got {tuple(target.shape)}")
    if logits.size(0) != target.size(0):
        raise ValueError("logits and target must have the same N dimension")

    # logsumexp: [N]
    log_denom = torch.logsumexp(logits, dim=-1)

    if ignore_index is None:
        valid_mask = torch.ones_like(target, dtype=torch.bool)
    else:
        valid_mask = target.ne(ignore_index)

    # 为了避免 ignore_index 是 -100 这类越界 id 时 gather 报错，
    # 先把无效 target 临时替换成 0。后面会用 valid_mask 把这些位置的 loss 置 0。
    safe_target = target.clone()
    safe_target = torch.where(valid_mask, safe_target, torch.zeros_like(safe_target))

    # 取出每个样本正确类别对应的 logit: z_{i,y_i}
    # correct_logits: [N]
    correct_logits = logits.gather(dim=1, index=safe_target.unsqueeze(1)).squeeze(1)

    # 每个位置的 loss: [N]
    loss = log_denom - correct_logits
    loss = torch.where(valid_mask, loss, torch.zeros_like(loss))

    if reduction == "none":
        return loss
    if reduction == "sum":
        return loss.sum()
    if reduction == "mean":
        valid_count = valid_mask.sum().clamp_min(1)
        return loss.sum() / valid_count
    raise ValueError(f"Unsupported reduction: {reduction}")

In [ ]:
batch_logits = torch.tensor(
    [
        [1.0, 2.0, 4.0, 0.5],
        [3.0, 0.0, 1.0, 2.0],
        [0.1, 0.2, 0.3, 0.4],
    ]
)
batch_target = torch.tensor([2, 0, 3])

manual_loss = cross_entropy_manual(batch_logits, batch_target)
torch_loss = nn.CrossEntropyLoss()(batch_logits, batch_target)

print("batch_logits.shape:", batch_logits.shape)
print("batch_target.shape:", batch_target.shape)
print("manual_loss:", manual_loss.item())
print("torch_loss: ", torch_loss.item())
print("allclose:", torch.allclose(manual_loss, torch_loss))

## 6. 为什么翻译模型要 reshape

在机器翻译里，模型不是只预测一个类别，而是对每个目标 token 位置都预测一个词表分布。

假设：

- `B = 2`，两个句子
- `T = 4`，每个目标句子 4 个位置
- `V = 6`，目标词表大小 6

模型输出：

$$logits \in \mathbb{R}^{B \times T \times V}$$

标签：

$$target \in \mathbb{R}^{B \times T}$$

`CrossEntropyLoss` 会把每个 token 位置当成一个分类样本，所以要展平：

$$[B, T, V] \rightarrow [B \cdot T, V]$$

$$[B, T] \rightarrow [B \cdot T]$$

In [ ]:
B, T, V = 2, 4, 6
logits = torch.randn(B, T, V)

# 这里 0 模拟 <pad>，其他数字模拟真实 token id。
tgt_pad_id = 0
tgt_output = torch.tensor(
    [
        [1, 2, 3, 0],
        [4, 5, 0, 0],
    ]
)

flat_logits = logits.reshape(-1, V)
flat_target = tgt_output.reshape(-1)

print("logits.shape:     ", logits.shape)
print("tgt_output.shape: ", tgt_output.shape)
print("flat_logits.shape:", flat_logits.shape)
print("flat_target.shape:", flat_target.shape)
print("flat_target:", flat_target.tolist())

## 7. `ignore_index` 为什么重要

目标序列通常会 padding 到同一长度，例如：

```text
句子 1: [1, 2, 3, <pad>]
句子 2: [4, 5, <pad>, <pad>]
```

如果不忽略 `<pad>`，模型会被迫学习预测大量 padding token。这样会污染训练目标。

所以 `train.py` 里用了：

```python
criterion = nn.CrossEntropyLoss(ignore_index=meta["tgt_pad_id"])
```

含义是：当 `target == tgt_pad_id` 时，该位置不参与 loss，也不参与平均。

In [ ]:
criterion_ignore_pad = nn.CrossEntropyLoss(ignore_index=tgt_pad_id)

loss_torch = criterion_ignore_pad(flat_logits, flat_target)
loss_manual = cross_entropy_manual(
    flat_logits,
    flat_target,
    ignore_index=tgt_pad_id,
    reduction="mean",
)

print("loss_torch: ", loss_torch.item())
print("loss_manual:", loss_manual.item())
print("allclose:", torch.allclose(loss_torch, loss_manual))

non_pad_tokens = flat_target.ne(tgt_pad_id).sum()
print("non_pad_tokens:", non_pad_tokens.item())

## 8. 看清楚每个 token 的 loss

设置 `reduction="none"` 可以看到每个位置的 loss。

对于被 `ignore_index` 忽略的 padding 位置，手写版本会把它们置为 0。PyTorch 的 `F.cross_entropy(..., reduction="none")` 对忽略位置也返回 0。

In [ ]:
token_loss_manual = cross_entropy_manual(
    flat_logits,
    flat_target,
    ignore_index=tgt_pad_id,
    reduction="none",
)
token_loss_torch = F.cross_entropy(
    flat_logits,
    flat_target,
    ignore_index=tgt_pad_id,
    reduction="none",
)

print("token_loss_manual flat:")
print(token_loss_manual)

print("\ntoken_loss_manual as [B, T]:")
print(token_loss_manual.reshape(B, T))

print("\nallclose:", torch.allclose(token_loss_manual, token_loss_torch))

## 9. 对齐 `train.py` 的 epoch 平均 loss 逻辑

你的 `train.py` 每个 batch 得到的是平均 token loss：

```python
loss = compute_loss(logits, tgt_output, criterion)
```

由于 `criterion = nn.CrossEntropyLoss(ignore_index=tgt_pad_id)` 默认 `reduction="mean"`，所以这个 `loss` 是当前 batch 内非 pad token 的平均 loss。

然后代码做：

```python
non_pad_tokens = tgt_output.ne(meta["tgt_pad_id"]).sum().item()
total_loss += loss.item() * non_pad_tokens
total_tokens += non_pad_tokens
epoch_loss = total_loss / total_tokens
```

这一步的目的：不同 batch 的非 pad token 数可能不同，不能简单平均每个 batch 的 loss。应该按 token 数加权平均。

In [ ]:
def compute_loss_like_train_py(logits, tgt_output, criterion):
    vocab_size = logits.size(-1)
    return criterion(logits.reshape(-1, vocab_size), tgt_output.reshape(-1))

loss_like_train = compute_loss_like_train_py(logits, tgt_output, criterion_ignore_pad)

# 直接把所有非 pad token 的逐位置 loss 求和再除以非 pad token 数。
token_loss_sum = token_loss_manual.sum()
token_loss_mean = token_loss_sum / non_pad_tokens

print("loss_like_train:", loss_like_train.item())
print("token_loss_mean:", token_loss_mean.item())
print("allclose:", torch.allclose(loss_like_train, token_loss_mean))

## 10. 常见错误 1：先 softmax 再传给 CrossEntropyLoss

错误写法：

```python
probs = torch.softmax(logits, dim=-1)
loss = nn.CrossEntropyLoss()(probs, target)
```

原因：`CrossEntropyLoss` 内部已经会做 `log_softmax`。如果你先 softmax，就相当于把概率当成 logits 再做一遍 `log_softmax`，数值含义变了。

正确写法：

```python
loss = nn.CrossEntropyLoss()(logits, target)
```

In [ ]:
wrong_probs = torch.softmax(flat_logits, dim=-1)

loss_correct = F.cross_entropy(flat_logits, flat_target, ignore_index=tgt_pad_id)
loss_wrong = F.cross_entropy(wrong_probs, flat_target, ignore_index=tgt_pad_id)

print("loss_correct_from_logits:", loss_correct.item())
print("loss_wrong_from_probs:  ", loss_wrong.item())

## 11. 常见错误 2：target 写成 one-hot

对于普通 `CrossEntropyLoss` 用法，target 应该是类别 id：

```python
target = torch.tensor([3, 5, 9])
```

不是：

```python
target = torch.tensor([
    [0, 0, 0, 1, 0, ...],
    [0, 0, 0, 0, 0, 1, ...],
])
```

在你的翻译任务里，`tgt_output` 里的每个元素就是目标词表中的 token id，所以它本来就是 `CrossEntropyLoss` 需要的格式。

## 12. 梯度直觉

对单个样本，交叉熵对 logits 的梯度是：

$$\frac{\partial Loss}{\partial z_i} = p_i - \mathbb{1}[i=y]$$

其中：

- `p_i` 是 softmax 后第 `i` 类的概率
- 如果 `i` 是正确类别，梯度是 `p_i - 1`，会推动正确类别 logit 变大
- 如果 `i` 不是正确类别，梯度是 `p_i`，会推动错误类别 logit 变小

这就是为什么分类模型通常直接输出 logits，然后用 CrossEntropyLoss 训练。

In [ ]:
grad_logits = single_logits.clone().detach().requires_grad_(True)
grad_loss = F.cross_entropy(grad_logits.unsqueeze(0), single_target.unsqueeze(0))
grad_loss.backward()

softmax_probs = torch.softmax(grad_logits.detach(), dim=-1)
expected_grad = softmax_probs.clone()
expected_grad[single_target] -= 1.0

print("autograd grad:", grad_logits.grad)
print("expected grad:", expected_grad)
print("allclose:", torch.allclose(grad_logits.grad, expected_grad))

## 13. 小结

结合本项目，记住这几条就够用：

1. `model.py` 输出的是 logits，不是概率。
2. `CrossEntropyLoss` 输入 logits，内部会做 `LogSoftmax + NLLLoss`。
3. target 是类别 id，不是 one-hot。
4. 翻译任务中每个目标 token 都是一个分类样本，所以要把 `[B, T, V]` 展平成 `[B*T, V]`。
5. `ignore_index=tgt_pad_id` 用来忽略 padding token。
6. epoch 平均 loss 应该按非 pad token 数加权，而不是简单平均 batch loss。